# Raspberry Pi AI Camera – Target Detection Training (Updated with Camera Specs)

This notebook uses the exact specifications of your camera to compute the subframe size that corresponds to a 30 cm × 30 cm target at 1 m distance. It then generates a synthetic dataset, trains a lightweight YOLOv8n model, and prepares it for deployment on the Raspberry Pi.

**Camera specifications (provided):**
- Sensor: Sony IMX500 (12.3 MP)
- Pixel size: 1.55 μm × 1.55 μm
- Horizontal FoV (full sensor): ≈ 66.3° (derived from diagonal FoV 78.3°)
- Filming resolution: 1280×720 (downscaled, so FoV is preserved)
- Effective distance: 1 m
- Target size: 30 cm × 30 cm (square)

## Step 1: Install Required Libraries

Run the following cell to install necessary packages. (If already installed, you can skip it.)

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# !pip install opencv-python numpy pillow tqdm ultralytics
!pip install opencv-python numpy pillow tqdm ultralytics --no-deps

# Verify the GPU CUDA Support:

In [1]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU


In [2]:
# ================== Camera Parameters ==================
EFFECTIVE_DISTANCE_M = 1.0          # distance to target (m)
TARGET_SIZE_CM = 30                 # side length of square target (cm)
CAMERA_HORIZONTAL_FOV_DEG = 66.3    # horizontal field of view at 720p (calculated from full sensor specs)
# ======================================================

# Compute R_s – the number of pixels covering a 30 cm target at 1 m
import numpy as np
import cv2
import os

def compute_Rs(horiz_fov_deg, distance_m, target_size_cm, image_width=1280):
    """Compute subframe size (pixels) that exactly covers a target of given size at given distance."""
    half_fov_rad = np.radians(horiz_fov_deg) / 2
    scene_width_m = 2 * distance_m * np.tan(half_fov_rad)
    pixel_size_m = scene_width_m / image_width
    target_size_m = target_size_cm / 100.0
    Rs = int(np.ceil(target_size_m / pixel_size_m))
    print(f"Horizontal FoV at {distance_m} m: {scene_width_m:.3f} m")
    print(f"Pixel size at {distance_m} m: {pixel_size_m*1000:.2f} mm")
    print(f"Target size {target_size_cm} cm → {Rs} pixels")
    return Rs

R_s = compute_Rs(CAMERA_HORIZONTAL_FOV_DEG, EFFECTIVE_DISTANCE_M, TARGET_SIZE_CM)
print(f"\nUsing subframe size: {R_s}×{R_s} pixels")

Horizontal FoV at 1.0 m: 1.306 m
Pixel size at 1.0 m: 1.02 mm
Target size 30 cm → 294 pixels

Using subframe size: 294×294 pixels


## Step 2: Generate Synthetic Dataset

**Important:** The dataset folder is completely reinitialised every time you run this cell. If you have important data, back it up first.

**Multi-target support:** define your target classes and where their images live in the configuration cell below (`TARGET_CLASSES`). Every class gets its own subfolder of target images and is assigned a class id automatically - the rest of the pipeline (labels, `dataset.yaml`, training) adapts to however many classes you define.

**Folder structure before running:**
```
project/
├── targets/
│   ├── target_a/      # target images for class "target_a" (RGBA PNG or RGB)
│   ├── target_b/      # target images for class "target_b"
│   └── ...            # add as many class folders as you need
├── backgrounds/      # background images (preferably 1280×720, but any size ≥ R_s)
└── this_notebook.ipynb
```

The generation includes:
- Fast OpenCV-based lighting augmentation
- Vectorised blending for target placement
- Configurable JPEG quality for faster writes
- Random target sizes between 5 cm and 30 cm (53 to R_s pixels)
- 80/20 train/val split
- Each positive sample draws from a randomly chosen class, with the correct class id written to its label

In [3]:
# ================== Multi-Target Configuration ==================
# Define how many target classes you have and where their images live.
# Add, remove, or rename entries freely - everything downstream (dataset
# generation, labels, dataset.yaml, and training) adapts automatically.
#
# Each key is the class name (used in dataset.yaml and shown in detections).
# Each value is the path to the folder containing that class's target images
# (RGBA PNGs work best; RGB images get an automatic alpha channel via
# background removal).
TARGET_CLASSES = {
    "target_a": "targets/target_a",
    "target_b": "targets/target_b",
    # "target_c": "targets/target_c",
}
# ===================================================================

CLASS_NAMES = list(TARGET_CLASSES.keys())
NUM_CLASSES = len(CLASS_NAMES)

print(f"Configured {NUM_CLASSES} target class(es): {CLASS_NAMES}")

Configured 2 target class(es): ['target_a', 'target_b']


In [ ]:
import cv2
import numpy as np
import random
import os
import shutil
from tqdm import tqdm

# ================== Generation Configuration ==================
NUM_POS = 10000          # number of sharp positive samples
NUM_NEG = 10000          # number of sharp negative samples
NUM_BLURRY_POS = 10000    # number of blurry positive samples
NUM_BLURRY_NEG = 10000    # number of blurry negative samples
BACKGROUND_FOLDER = "backgrounds"
OUTPUT_DIR = "dataset"
TRAIN_RATIO = 0.8        # 80% train / 20% validation
JPEG_QUALITY = 75
# =============================================================

# Delete and recreate dataset
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(f"{OUTPUT_DIR}/images/train", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/images/val", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels/train", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels/val", exist_ok=True)

# -------------------- Helper Functions (unchanged) --------------------
def load_target_images(folder):
    """Load images and ensure they have a valid alpha channel."""
    images = []
    for fname in os.listdir(folder):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        path = os.path.join(folder, fname)
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if img is None:
            continue
        if len(img.shape) == 3 and img.shape[2] == 4:
            images.append(img)
            continue
        if len(img.shape) == 3 and img.shape[2] == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            _, alpha = cv2.threshold(gray, 250, 255, cv2.THRESH_BINARY_INV)
            alpha = cv2.medianBlur(alpha, 5)
            rgba = np.dstack((img, alpha))
            images.append(rgba)
            print(f"Auto-alpha created for {fname}")
            continue
        print(f"Warning: {fname} skipped (unsupported format).")
    return images

def load_background_images(folder):
    images = []
    for fname in os.listdir(folder):
        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            img = cv2.imread(os.path.join(folder, fname))
            if img is not None:
                images.append(img)
    return images

def random_lighting_fast(img):
    brightness = random.uniform(-30, 30)
    img = cv2.add(img, brightness)
    contrast = random.uniform(0.7, 1.3)
    img = cv2.convertScaleAbs(img, alpha=contrast, beta=0)
    if random.random() < 0.5:
        gamma = random.uniform(0.8, 1.2)
        look_up = np.array([((i / 255.0) ** gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
        img = cv2.LUT(img, look_up)
    return img

def transform_target(target, target_size):
    h, w = target.shape[:2]
    scale = target_size / max(h, w)
    new_w = int(w * scale)
    new_h = int(h * scale)
    target = cv2.resize(target, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    angle = random.uniform(0, 360)
    M = cv2.getRotationMatrix2D((new_w/2, new_h/2), angle, 1.0)
    target = cv2.warpAffine(target, M, (new_w, new_h),
                            flags=cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_CONSTANT,
                            borderValue=(0,0,0,0))
    if random.random() < 0.5:
        shear_x = random.uniform(-0.1, 0.1)
        shear_y = random.uniform(-0.1, 0.1)
        M = np.float32([[1, shear_x, 0], [shear_y, 1, 0]])
        target = cv2.warpAffine(target, M, (new_w, new_h),
                                flags=cv2.INTER_LINEAR,
                                borderMode=cv2.BORDER_CONSTANT,
                                borderValue=(0,0,0,0))
    return target

def place_target_on_background(target, bg_subframe):
    th, tw = target.shape[:2]
    bh, bw = bg_subframe.shape[:2]
    x = random.randint(-tw, bw - 1)
    y = random.randint(-th, bh - 1)
    src_x1 = max(0, -x)
    src_y1 = max(0, -y)
    src_x2 = min(tw, bw - x)
    src_y2 = min(th, bh - y)
    dst_x1 = max(0, x)
    dst_y1 = max(0, y)
    if src_x2 <= src_x1 or src_y2 <= src_y1:
        return bg_subframe, None
    target_rgb = target[src_y1:src_y2, src_x1:src_x2, :3]
    alpha = target[src_y1:src_y2, src_x1:src_x2, 3:4] / 255.0
    roi = bg_subframe[dst_y1:dst_y1 + (src_y2 - src_y1),
                      dst_x1:dst_x1 + (src_x2 - src_x1)]
    bg_subframe[dst_y1:dst_y1 + (src_y2 - src_y1),
                dst_x1:dst_x1 + (src_x2 - src_x1)] = \
        (alpha * target_rgb + (1 - alpha) * roi).astype(np.uint8)
    bbox = [dst_x1, dst_y1, src_x2 - src_x1, src_y2 - src_y1]
    return bg_subframe, bbox

def save_sample(image, bbox, idx, split, class_id=0):
    img_path = f"{OUTPUT_DIR}/images/{split}/img_{idx:06d}.jpg"
    cv2.imwrite(img_path, image, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
    if bbox is not None:
        h, w = image.shape[:2]
        xc = (bbox[0] + bbox[2]/2) / w
        yc = (bbox[1] + bbox[3]/2) / h
        bw = bbox[2] / w
        bh = bbox[3] / h
        label_path = f"{OUTPUT_DIR}/labels/{split}/img_{idx:06d}.txt"
        with open(label_path, 'w') as f:
            f.write(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

# -------------------- NEW: Blur functions for drone vibration --------------------
def apply_motion_blur(img, max_kernel_size=15):
    """
    Apply a motion blur kernel with random direction and length.
    """
    # Random kernel size (odd) between 3 and max_kernel_size
    size = random.randrange(3, max_kernel_size + 1, 2)
    # Random angle in degrees
    angle = random.uniform(0, 360)
    # Create the motion blur kernel
    kernel = np.zeros((size, size), dtype=np.float32)
    # Set the center line with ones along the angle
    center = size // 2
    # Use line equation: y = tan(angle) * (x - center) + center
    # We'll fill pixels that are close to the line
    for i in range(size):
        for j in range(size):
            # Distance from point (i,j) to the line through center with given angle
            # Line in parametric form: direction vector (cos, sin)
            dx = i - center
            dy = j - center
            # Project onto direction
            dir_x = np.cos(np.radians(angle))
            dir_y = np.sin(np.radians(angle))
            proj = dx * dir_x + dy * dir_y
            # Perpendicular distance
            perp = np.sqrt((dx - proj * dir_x)**2 + (dy - proj * dir_y)**2)
            # If within 0.5 pixels, mark as 1
            if perp < 0.5 and abs(proj) <= center:
                kernel[i, j] = 1
    # Normalize kernel
    if np.sum(kernel) > 0:
        kernel = kernel / np.sum(kernel)
    else:
        # fallback to Gaussian if no line (shouldn't happen)
        kernel = np.ones((size, size), dtype=np.float32) / (size * size)
    blurred = cv2.filter2D(img, -1, kernel)
    return blurred

def apply_gaussian_blur(img, max_kernel_size=15):
    """Apply a random Gaussian blur."""
    size = random.randrange(3, max_kernel_size + 1, 2)
    sigma = random.uniform(0.5, 3.0)
    return cv2.GaussianBlur(img, (size, size), sigma)

def apply_blur(img):
    """Randomly choose between motion blur and Gaussian blur."""
    if random.random() < 0.5:
        return apply_motion_blur(img)
    else:
        return apply_gaussian_blur(img)

# -------------------- Load Images --------------------
print("Loading target images...")
class_targets = {}
CLASS_NAMES = ["target_a", "target_b"]  # define your classes
# You need to define TARGET_CLASSES dict mapping class name to folder
# For example: TARGET_CLASSES = {"target_a": "targets/target_a", "target_b": "targets/target_b"}
# If not defined, you can set it here or use a single folder.
# For simplicity, assuming you have a single folder "targets" with multiple classes? 
# I'll adapt to your existing code: you already have class_targets dictionary.
# I'll assume you have TARGET_CLASSES defined elsewhere, but to be safe:
if 'TARGET_CLASSES' not in globals():
    # Fallback: if only one class, use default folder "targets"
    TARGET_CLASSES = {"target": "targets"}

for class_id, class_name in enumerate(CLASS_NAMES):
    folder = TARGET_CLASSES.get(class_name, None)
    if folder is None:
        print(f"Warning: no folder for class {class_name}, skipping.")
        continue
    imgs = load_target_images(folder)
    if imgs:
        class_targets[class_id] = imgs
    print(f"  [{class_id}] {class_name}: loaded {len(imgs)} image(s) from '{folder}'")

if not class_targets:
    raise Exception("No valid target images found. Check your TARGET_CLASSES.")

total_targets = sum(len(v) for v in class_targets.values())
print(f"Loaded {total_targets} target image(s) across {len(class_targets)} class(es).")

print("Loading background images...")
backgrounds = load_background_images(BACKGROUND_FOLDER)
print(f"Loaded {len(backgrounds)} background images.")

if not backgrounds:
    raise Exception("No background images found.")

backgrounds = [bg for bg in backgrounds if bg.shape[0] >= R_s and bg.shape[1] >= R_s]
if not backgrounds:
    raise Exception(f"No background images are at least {R_s}×{R_s} pixels.")
print(f"After filtering, {len(backgrounds)} backgrounds remain.")

# -------------------- Helper to generate samples (with optional blur) --------------------
def generate_samples(num_pos, num_neg, idx_start, apply_blur_flag=False, desc_prefix="Generating"):
    pos_count, idx = 0, idx_start
    total_samples = num_pos + num_neg
    pbar = tqdm(total=total_samples, desc=desc_prefix)
    # Generate positives
    while pos_count < num_pos:
        bg = random.choice(backgrounds)
        x = random.randint(0, bg.shape[1] - R_s)
        y = random.randint(0, bg.shape[0] - R_s)
        subframe = bg[y:y+R_s, x:x+R_s].copy()
        subframe = random_lighting_fast(subframe)

        class_id = random.choice(list(class_targets.keys()))
        target = random.choice(class_targets[class_id])
        min_px = int(round(5.0 / (TARGET_SIZE_CM / R_s)))
        target_size = random.randint(min_px, R_s)
        target = transform_target(target, target_size)

        composite, bbox = place_target_on_background(target, subframe)
        if bbox is None:
            continue

        if apply_blur_flag:
            composite = apply_blur(composite)

        split = 'train' if random.random() < TRAIN_RATIO else 'val'
        save_sample(composite, bbox, idx, split, class_id)
        idx += 1
        pos_count += 1
        pbar.update(1)

    # Generate negatives
    neg_count = 0
    while neg_count < num_neg:
        bg = random.choice(backgrounds)
        x = random.randint(0, bg.shape[1] - R_s)
        y = random.randint(0, bg.shape[0] - R_s)
        subframe = bg[y:y+R_s, x:x+R_s].copy()
        subframe = random_lighting_fast(subframe)
        if apply_blur_flag:
            subframe = apply_blur(subframe)
        split = 'train' if random.random() < TRAIN_RATIO else 'val'
        save_sample(subframe, None, idx, split)
        idx += 1
        neg_count += 1
        pbar.update(1)
    pbar.close()
    return idx

# -------------------- Generate Sharp Samples --------------------
print("\nGenerating sharp samples...")
idx = generate_samples(NUM_POS, NUM_NEG, 0, apply_blur_flag=False, desc_prefix="Sharp samples")

# -------------------- Generate Blurry Samples --------------------
if NUM_BLURRY_POS > 0 or NUM_BLURRY_NEG > 0:
    print("\nGenerating blurry samples (simulating drone vibration)...")
    idx = generate_samples(NUM_BLURRY_POS, NUM_BLURRY_NEG, idx, apply_blur_flag=True, desc_prefix="Blurry samples")

print(f"\nDone. Total images generated: {idx} (sharp positives {NUM_POS}, sharp negatives {NUM_NEG}, blurry positives {NUM_BLURRY_POS}, blurry negatives {NUM_BLURRY_NEG}).")

## Step 3: Train YOLOv8n Model

Now we train a lightweight YOLOv8n model on the synthetic dataset. Adjust the number of epochs and batch size according to your hardware.

In [ ]:
from ultralytics import YOLO
import yaml

# Write dataset configuration
data_yaml = f"""
path: {OUTPUT_DIR}
train: images/train
val: images/val

nc: {NUM_CLASSES}
names: {CLASS_NAMES}
"""
with open('dataset.yaml', 'w') as f:
    f.write(data_yaml)

# Load pretrained YOLOv8n (you can also use YOLOv8s or YOLOv8m for more accuracy)
model = YOLO('yolov8n.pt')

# Train
model.train(
    data='dataset.yaml',
    epochs=300,
    imgsz=R_s,
    batch=32,
    workers=8,
    device=0,
    pretrained=True,
    optimizer='Adam',
    lr0=0.001,
    augment=True,
    project='runs/train',
    name='target_detector'
)

# Evaluate on validation set
metrics = model.val()
print(metrics)

# Export to ONNX for faster inference on Raspberry Pi
model.export(format='onnx', imgsz=R_s)

## Step 4: Deployment Notes

After training, the best model is saved in `runs/train/target_detector/weights/best.pt`. You can copy this file to your Raspberry Pi along with the exported ONNX version (`best.onnx`) for inference.

### Inference pipeline outline on the Pi:
1. Capture a 720p frame from the camera.
2. Use the aligned ToF depth map to create a binary mask of pixels within 1 m.
3. Slide a window of size \(R_s \times R_s\) over the masked region (e.g., grid or overlapping sliding window).
4. For each subframe, run the model:
   ```python
   results = model(subframe, imgsz=R_s)
   ```
5. If a target is detected, map its bounding box back to the full frame and then to real‑world coordinates using the depth information.

### Performance tips:
- Use the ONNX model with `onnxruntime` (CPU or GPU) for faster inference.
- Consider using a smaller model like YOLOv8n (already used) or YOLOv8n‑quantized.
- Reduce the input resolution if needed (but retrain with that resolution).
- Leverage the Pi's GPU (if available) via OpenCV's CUDA or the `libcamera` stack.

---
**End of notebook** – you now have a complete pipeline from synthetic data generation to training and deployment planning.

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

# --- Load your trained YOLO model ---
model = YOLO('paste in the path provided by the training block')  # or use the ONNX model
# For ONNX: model = YOLO('best.onnx')

# --- Preprocessing function (same as before) ---
def preprocess_for_yolo(image_path, distance_cm, R_s, HFOV_DEG=66.26, VFOV_DEG=52.02):
    # Load image (should be 1280x720)
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Compute crop size for 30x30 cm area
    crop_w = int(1280 * (30 / (2 * distance_cm * np.tan(np.radians(HFOV_DEG)/2))))
    crop_h = int(720 * (30 / (2 * distance_cm * np.tan(np.radians(VFOV_DEG)/2))))
    h, w = img.shape[:2]
    start_x = (w - crop_w) // 2
    start_y = (h - crop_h) // 2
    cropped = img_rgb[start_y:start_y+crop_h, start_x:start_x+crop_w]

    # Resize to training size
    resized = cv2.resize(cropped, (R_s, R_s), interpolation=cv2.INTER_AREA)
    return resized

# --- Run inference ---
R_s = 224  # must match training
distance_cm = 100  # replace with actual distance
img_processed = preprocess_for_yolo(r'<Some file here :P>', distance_cm, R_s)

results = model(img_processed)  # results is a list of Results objects
# Show results
results[0].show()  # displays the image with bounding boxes


0: 320x320 1 target_a, 7.1ms
Speed: 0.7ms preprocess, 7.1ms inference, 1.3ms postprocess per image at shape (1, 3, 320, 320)
